# Plot B: t-SNE + BV distance to centroid

**Purpose:** CCN 2026 analysis using **BabyView (BV) only**: for each category, compute every BV exemplar’s Euclidean distance to the **BV category centroid** in embedding space, then run **t-SNE on BV exemplars plus the BV centroid** (no THINGS exemplars). Exemplars are colored by distance to that centroid. You can run CLIP and DINOv3 in one pass; saved files and folders use the stem `bv-to-bv-centroid_distance_<model>` (see `output_metric_slug` in Parameters).

**Outputs:**
1. Per-category summary CSV (`mean_bv_to_bv_centroid`, `std_bv_to_bv_centroid`, `n_bv`).
2. Optional per-exemplar CSV with `dist_to_bv_centroid` (and ids) per crop.
3. Per-category t-SNE PNG/PDF under `{prefix}_tsne_by_category` (same figure size and marker styling as the previous joint plot).
3b. Optional **joint** t-SNE (`tsne_joint_*` in the same folder): 2–4 categories **plotted** in one figure. By default (`tsne_joint_tsne_background = "all_valid_categories"`), t-SNE is fit on **all** precision-valid BV exemplars across every common category (full context); only the selected categories are drawn (centroid markers = mean of their 2D positions). Set `tsne_joint_tsne_background = "highlight_only"` to fit t-SNE on the highlighted categories only (previous behavior: exemplars + centroid rows). After the side-by-side cell, matching **CLIP vs DINOv3 joint** figures go under `{output_metric_slug}_clip_vs_dinov3_tsne_joint_side_by_side` (`tsne_joint_side_by_side_*` and `*_aligned`). When `tsne_joint_tsne_background` is `"all_valid_categories"`, an extra **`tsne_joint_*_local*`** variant is also saved: same four categories, but t-SNE fit **only** on those categories (less compressed; not comparable to global layout).
4. Ranked semantic barplot and optional histogram of category-level spread; montage generation is turned off in this notebook (helper functions remain for other use).

## Parameters (set these before running)

Adjust paths, embedding list (`clip` / `dinov3`), t-SNE selection, and output dir. This notebook is **BV-only** (no THINGS loading or joint t-SNE).

In [1]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from scipy import stats
from tqdm import tqdm
from PIL import Image, ImageDraw, ImageFont

SCRIPT_DIR = Path(".").resolve()

# --- Config (equivalent to argparse) ---
embeddings_to_run = ["dinov3"]  # minimal: DINOv3 joint local; add "clip" for full two-model run
_run_id = datetime.now().strftime("%Y%m%d")
out_dir = SCRIPT_DIR / f"plotB_tsne_distance_to_centroid_outputs_{_run_id}"
# Stem for CSVs, figures, and subfolders (BV exemplar → BV centroid distance)
output_metric_slug = "bv-to-bv-centroid_distance"
save_exemplar_csv = True

filter_threshold = 0.27
project_root = SCRIPT_DIR.parent.parent  # object-detection repo root
metadata_csv = project_root / f"frame_data/merged_frame_detections_with_metadata_filtered-{filter_threshold}.csv"

# t-SNE selection mode: top-K and bottom-K categories by BV-centroid distance
tsne_select_top_bottom_by_distance = True
tsne_top_k = 5
tsne_bottom_k = 5
# metric option kept as BV-only distance
tsne_rank_metric = "mean_bv_to_bv_centroid"

tsne_one_per_semantic = False
tsne_semantic_source_csv = project_root / "data/long_tailed_dist_prop_included_categories.csv"
cdi_semantic_csv_for_joint_colors = tsne_semantic_source_csv  # Plot B bar / Plot C CDI hex mapping
tsne_category = []
tsne_use_plotA_selection = True
tsne_use_plotA_exemplars = True
plotA_selected_csv = SCRIPT_DIR / "plotA_category_montages_low_to_high/plotA_selected_categories_low_to_high_variability.csv"
plotA_sampled_exemplars_csv = project_root / "annotation/sampled_object_crops_100_bucket_assignments_100ex_8subj_per_video_cap_babyview_only.csv"
plotA_n_exemplars = 25
tsne_all_categories = False
tsne_perplexity = 28.0
tsne_top12_by_similarity = False
tsne_zscore = True

# Joint t-SNE: 2–4 categories in one embedding (shared axes; relative spread comparable across cats)
tsne_joint_multi_category = True
# None = one auto panel: first min(4, len(categories_to_plot)) categories (same order as selection).
# Two joint panels: 3-category and 4-category (DINOv3 joint outputs only; see tsne_joint_save_embeddings).
tsne_joint_category_groups = [["watch", "shower", "bird"], ["watch", "shower", "bird", "pony"]]
# Joint t-SNE fit scope: "all_valid_categories" = t-SNE on all precision-valid exemplars (all common cats), plot only the joint group; # "highlight_only" = t-SNE only on those categories (exemplars + centroid rows in the fit).
tsne_joint_tsne_background = "all_valid_categories"
tsne_joint_only_local = True  # skip global joint t-SNE; save tsne_joint_*_local.png only (faster)
# Joint tsne_joint_* panels only for these embeddings (None or [] = all in embeddings_to_run).
tsne_joint_save_embeddings = ["dinov3"]
# "hue_saturation" = category hue + saturation ~ distance; "magma" = single colormap.
tsne_joint_color_mode = "hue_saturation"
tsne_joint_sat_gamma = 0.42  # lower → more saturation spread across distances
tsne_joint_sat_floor = 0.12  # minimum saturation
# CLIP vs DINO joint side-by-side exports; False = per-model joint only.
tsne_joint_clip_vs_dinov3_side_by_side = False
_joint_tsne_groups_cache = []  # populated by t-SNE cell for CLIP vs DINOv3 joint side-by-side

# Optional extras (disabled for notebook 2; t-SNE only)
rdm_top12_and_things = False
rdm_max_things = 50
montage_farthest = False
montage_top12_by_similarity = False
montage_n = 12
montage_max_categories = 0  # 0 = all

cropped_dir = Path("/data2/dataset/babyview/868_hours/outputs/yoloe_cdi_all_cropped_by_class")
montage_cell_size = (128, 128)
montage_cols = 4

In [2]:
# Original (non-grouped) per-image embeddings; only rows in clip-filtered metadata are used (filter_threshold)
BV_EMBEDDINGS_BASE = Path("/data2/dataset/babyview/868_hours/outputs/yoloe_cdi_embeddings")
BV_ORIGINAL_EMBEDDINGS_DIRS = {
    "clip": BV_EMBEDDINGS_BASE / "clip_embeddings_new",
    "dinov3": BV_EMBEDDINGS_BASE / "facebook_dinov3-vitb16-pretrain-lvd1689m",
}
# Match preprint-2026 category selection (129 categories)
CATEGORIES_FILE = SCRIPT_DIR / "../../data/included_categories.txt"
PER_CLASS_VALIDATION_CSV = SCRIPT_DIR / "../../annotation/per_class_validation_data.csv"
PER_FILE_PRECISION_CSV = SCRIPT_DIR / "../../annotation/per_file_precision_data.csv"
PRECISION_THRESHOLD = 0.6
EXCLUDED_SUBJECT = "00270001"
MIN_EXEMPLARS_BV = 2
# All-frames CSV with image-text (or image-to-centroid) similarity per (category, embedding); used for top-12 BV montage
SIMILARITY_CSV = BV_EMBEDDINGS_BASE / "clip_image_embeddings_filtered_all_frames_exclude-people_exclude-subject-00270001.csv"

## Helpers: crop dir, exemplar→crop lookup, montage

In [3]:
def get_category_crop_dir(cropped_dir, cat_name):
    """Return Path to category subfolder (match by name, case-insensitive)."""
    cat_lower = cat_name.strip().lower()
    direct = cropped_dir / cat_name
    if direct.exists() and direct.is_dir():
        return direct
    for p in cropped_dir.iterdir():
        if p.is_dir() and p.name.lower() == cat_lower:
            return p
    return cropped_dir / cat_name


def build_exemplar_to_crop_lookup(metadata_csv, usecols=None, chunksize=500_000):
    """Build (class_name, subject_id, age_mo) -> original_embedding_name (stem for crop)."""
    if usecols is None:
        usecols = ["class_name", "subject_id", "age_mo", "original_embedding_name"]
    lookup = {}
    for chunk in tqdm(
        pd.read_csv(metadata_csv, usecols=usecols, chunksize=chunksize,
                    dtype={"subject_id": str, "class_name": str},
                    na_values=[], keep_default_na=False),
        desc="Metadata chunks", unit="chunk",
    ):
        chunk = chunk.dropna(subset=["class_name", "subject_id", "age_mo", "original_embedding_name"])
        chunk["subject_id_norm"] = chunk["subject_id"].str.strip().str.lstrip("S")
        chunk["age_mo_int"] = pd.to_numeric(chunk["age_mo"], errors="coerce").fillna(-1).astype(int)
        chunk = chunk[chunk["age_mo_int"] >= 0]
        for _, row in chunk.iterrows():
            key = (str(row["class_name"]).strip().lower(), row["subject_id_norm"], row["age_mo_int"])
            if key not in lookup:
                stem = str(row["original_embedding_name"]).strip()
                if stem.endswith(".npy"):
                    stem = stem[:-4]
                lookup[key] = stem
    return lookup


def make_montage_with_labels(images, labels, n_cols, cell_size, label_height=24):
    """Tile PIL images into a grid and draw distance label below each cell."""
    if not images:
        return None
    n = len(images)
    n_rows = (n + n_cols - 1) // n_cols
    total_h = n_rows * (cell_size[1] + label_height)
    total_w = n_cols * cell_size[0]
    out = Image.new("RGB", (total_w, total_h), (255, 255, 255))
    draw = ImageDraw.Draw(out)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 12)
    except Exception:
        font = ImageFont.load_default()
    for idx, img in enumerate(images):
        row, col = idx // n_cols, idx % n_cols
        if img.size != (cell_size[0], cell_size[1]):
            img = img.resize((cell_size[0], cell_size[1]), Image.Resampling.LANCZOS)
        r0 = row * (cell_size[1] + label_height)
        c0 = col * cell_size[0]
        out.paste(img, (c0, r0))
        if idx < len(labels):
            draw.text((c0 + 2, r0 + cell_size[1] + 2), f"d={labels[idx]}", fill=(0, 0, 0), font=font)
    return out

## Load categories and BV embeddings

In [4]:
def load_allowed_categories():
    allowed = None
    if CATEGORIES_FILE.exists():
        with open(CATEGORIES_FILE) as f:
            allowed = set(line.strip().lower() for line in f if line.strip())
    if not PER_CLASS_VALIDATION_CSV.exists():
        return allowed
    val_df = pd.read_csv(PER_CLASS_VALIDATION_CSV, usecols=["class", "precision"])\
        .dropna(subset=["class", "precision"]).copy()
    val_df["class"] = val_df["class"].astype(str).str.strip().str.lower()
    val_df["precision"] = pd.to_numeric(val_df["precision"], errors="coerce")
    precision_allowed = set(val_df.loc[val_df["precision"] > PRECISION_THRESHOLD, "class"])
    if allowed is None:
        return precision_allowed
    return allowed & precision_allowed


def parse_confidence_from_stem(stem):
    parts = str(stem).split("_")
    if len(parts) < 2:
        return np.nan
    try:
        return float(parts[1])
    except ValueError:
        return np.nan


def load_valid_file_stems(per_file_precision_csv, threshold=0.6):
    if per_file_precision_csv is None or not Path(per_file_precision_csv).exists():
        return None
    df = pd.read_csv(per_file_precision_csv, usecols=["filename", "class", "precision"]).dropna().copy()
    df["class"] = df["class"].astype(str).str.strip().str.lower()
    df["precision"] = pd.to_numeric(df["precision"], errors="coerce")
    df = df[df["precision"] > threshold]
    valid = {}
    for fname_raw, cat_raw, _prec in df[["filename", "class", "precision"]].itertuples(index=False, name=None):
        cat = str(cat_raw).strip().lower()
        stem = Path(str(fname_raw)).stem
        bucket = valid.setdefault(cat, set())
        bucket.add(stem)
    return valid


def load_plotA_selected_categories(selected_csv):
    if selected_csv is None or not Path(selected_csv).exists():
        return []
    df = pd.read_csv(selected_csv)
    if "category" not in df.columns:
        return []
    return [str(c).strip().lower() for c in df["category"].tolist() if str(c).strip()]


def load_sampled_high_confidence_stems(sampled_csv, categories_set, n_exemplars, valid_file_stems=None):
    if sampled_csv is None or not Path(sampled_csv).exists():
        return {}
    df = pd.read_csv(sampled_csv)
    df = df.dropna(subset=["category", "stem"]).copy()
    df["category_norm"] = df["category"].astype(str).str.strip().str.lower()
    if categories_set is not None:
        allowed = {str(c).strip().lower() for c in categories_set}
        df = df[df["category_norm"].isin(allowed)]
    if valid_file_stems is not None:
        df = df[df.apply(lambda r: str(r["stem"]).strip() in valid_file_stems.get(str(r["category_norm"]).strip().lower(), set()), axis=1)]
    df["confidence"] = df["stem"].map(parse_confidence_from_stem)
    df = df.dropna(subset=["confidence"])
    stems_by_cat = {}
    for cat, g in df.groupby("category_norm"):
        g = g.sort_values("confidence", ascending=False).head(n_exemplars)
        stems_by_cat[cat] = [str(s).strip() for s in g["stem"].tolist()]
    return stems_by_cat


def load_filtered_metadata(metadata_csv, excluded_subject=None):
    """Load filtered metadata; return DataFrame with original_embedding_name, class_name (lower), subject_id_norm, age_mo (int)."""
    df = pd.read_csv(metadata_csv, usecols=["original_embedding_name", "class_name", "subject_id", "age_mo"],
                     dtype={"subject_id": str, "class_name": str}, na_values=[], keep_default_na=False)
    df = df.dropna(subset=["original_embedding_name", "class_name"])
    df["class_name"] = df["class_name"].str.strip().str.lower()
    df["subject_id_norm"] = df["subject_id"].astype(str).str.strip().str.lstrip("S")
    df["age_mo"] = pd.to_numeric(df["age_mo"], errors="coerce").fillna(-1).astype(int)
    if excluded_subject:
        df = df[df["subject_id_norm"] != excluded_subject]
    df = df.drop_duplicates(subset=["original_embedding_name"], keep="first")
    return df


def select_one_category_per_semantic(semantic_csv_path):
    """Pick one representative category per CDI semantic group (highest proportion)."""
    sem_df = pd.read_csv(semantic_csv_path)
    required_cols = {"category", "proportion", "cdi_semantic"}
    if not required_cols.issubset(set(sem_df.columns)):
        raise ValueError(f"Semantic CSV missing required columns: {required_cols}")
    sem_df["category"] = sem_df["category"].astype(str).str.strip().str.lower()
    sem_df["cdi_semantic"] = sem_df["cdi_semantic"].astype(str).str.strip().str.lower()
    sem_df["proportion"] = pd.to_numeric(sem_df["proportion"], errors="coerce").fillna(0.0)

    # Highest-frequency category represents each semantic bin.
    idx = sem_df.groupby("cdi_semantic")["proportion"].idxmax()
    picked = sem_df.loc[idx].sort_values("cdi_semantic")
    return picked[["cdi_semantic", "category", "proportion"]].reset_index(drop=True)


def load_bv_embeddings_from_filtered(embedding_type, metadata_csv, allowed_categories, excluded_subject, min_exemplars=2):
    """Load BV from original (non-grouped) embedding dir, only for rows in filtered metadata.
    Returns category_embeddings, category_exemplar_ids with ids = [(subject_id, age_mo, stem), ...]."""
    embeddings_dir = Path(BV_ORIGINAL_EMBEDDINGS_DIRS[embedding_type])
    if not embeddings_dir.exists():
        raise FileNotFoundError(f"BV embeddings dir not found: {embeddings_dir}")
    meta = load_filtered_metadata(metadata_csv, excluded_subject=excluded_subject)
    cat_to_rows = meta.groupby("class_name", sort=False).apply(
        lambda g: list(zip(g["original_embedding_name"], g["subject_id_norm"], g["age_mo"]))
    ).to_dict()
    category_embeddings = {}
    category_exemplar_ids = {}
    for cat_name, rows in cat_to_rows.items():
        if allowed_categories is not None and cat_name not in allowed_categories:
            continue
        embs, ids = [], []
        cat_folder = embeddings_dir / cat_name
        if not cat_folder.exists():
            for d in embeddings_dir.iterdir():
                if d.is_dir() and d.name.lower() == cat_name:
                    cat_folder = d
                    break
        for emb_name, sid, age_mo in rows:
            path = cat_folder / emb_name
            if not path.exists() and not str(emb_name).endswith(".npy"):
                path = cat_folder / f"{emb_name}.npy"
            if not path.exists():
                continue
            try:
                e = np.load(path)
                e = np.asarray(e, dtype=np.float64).flatten()
                embs.append(e)
                stem = str(emb_name).rstrip(".npy") if str(emb_name).endswith(".npy") else str(emb_name)
                ids.append((sid, age_mo, stem))
            except Exception:
                continue
        if len(embs) >= min_exemplars:
            category_embeddings[cat_name] = np.array(embs)
            category_exemplar_ids[cat_name] = ids
    return category_embeddings, category_exemplar_ids



## Compute distances and build summary (BV exemplar → BV centroid)

In [5]:
def run_distances(bv_embeddings, bv_ids, categories_common):
    summary_rows = []
    exemplar_rows = []
    for cat in tqdm(categories_common, desc="Categories"):
        bv_X = bv_embeddings[cat]
        bv_id_list = bv_ids[cat]

        bv_centroid = bv_X.mean(axis=0)
        dist_to_bv = np.linalg.norm(bv_X - bv_centroid, axis=1)

        n_bv = bv_X.shape[0]
        mean_d_bv = float(np.mean(dist_to_bv))
        std_d_bv = float(np.std(dist_to_bv))
        summary_rows.append({
            "category": cat,
            "mean_bv_to_bv_centroid": mean_d_bv,
            "std_bv_to_bv_centroid": std_d_bv,
            "n_bv": n_bv,
        })
        for i, id_entry in enumerate(bv_id_list):
            sid, age_mo = id_entry[0], id_entry[1]
            stem = id_entry[2] if len(id_entry) > 2 else None
            exemplar_rows.append({
                "category": cat,
                "subject_id": sid,
                "age_mo": age_mo,
                "dist_to_bv_centroid": float(dist_to_bv[i]),
                "stem": stem,
            })
    return summary_rows, exemplar_rows

## t-SNE for one category (BV exemplars + BV centroid; colored by distance to BV centroid)

In [ ]:
# Typography for all t-SNE scatter figures (bold text, readable sizes)
_TSNE_TITLE_FS = 15
_TSNE_AXIS_FS = 16
_TSNE_TICK_FS = 14
_TSNE_CBAR_LABEL_FS = 15
_TSNE_CBAR_TICK_FS = 13
_TSNE_LEGEND_FS = 13
_JOINT_TSNE_COLORS = ["#c0392b", "#2980b9", "#16a085", "#8e44ad"]
_JOINT_CENTROID_S_FILLED = 280
_JOINT_CENTROID_S_RING = 410
# Match 03_plotC_knn_diversity 2x2 panel width (figsize=(18, 13)) for horizontal figure assembly.
_TSNE_JOINT_FIG_W_MATCH_PLOTC_2X2 = 18.0
_TSNE_JOINT_SINGLE_FIGSIZE = (_TSNE_JOINT_FIG_W_MATCH_PLOTC_2X2 / 2, 6.6)
_TSNE_JOINT_CLIP_DINO_FIGSIZE = (_TSNE_JOINT_FIG_W_MATCH_PLOTC_2X2, 6.6)


# CDI semantic hex — match Plot B ranked bar + Plot C 2x2 (same dict).
_CDI_SEMANTIC_COLORS_JOINT = {
    "animals": "#4DB8A8",
    "body_parts": "#E87A5F",
    "clothing": "#9B7EC8",
    "food_drink": "#E8A54C",
    "furniture_rooms": "#6BAB7A",
    "household": "#D97B9E",
    "outside": "#5B9BD5",
    "people": "#E8C44C",
    "toys": "#B07CC8",
    "vehicles": "#6BA3D5",
    "other": "#8B9A9E",
}
# Bird vs pony: two similar teal-greens (both animals) but separable in the legend and on the plot.
_JOINT_ANIMAL_PAIR_HEX = {"bird": "#2E9B86", "pony": "#6FD0BE"}
_joint_cdi_sem_cache = {}


def _joint_load_category_to_cdi_semantic():
    global _joint_cdi_sem_cache
    if _joint_cdi_sem_cache:
        return _joint_cdi_sem_cache
    csv_path = globals().get("cdi_semantic_csv_for_joint_colors")
    if csv_path is None:
        pr = globals().get("project_root")
        if pr is None:
            return {}
        csv_path = Path(pr) / "data/long_tailed_dist_prop_included_categories.csv"
    else:
        csv_path = Path(csv_path)
    if not csv_path.is_file():
        return {}
    import pandas as pd
    sem_df = pd.read_csv(csv_path, usecols=["category", "cdi_semantic"]).dropna()
    sem_df["category"] = sem_df["category"].astype(str).str.strip().str.lower()
    sem_df["cdi_semantic"] = sem_df["cdi_semantic"].astype(str).str.strip().str.lower()
    sem_df = sem_df.drop_duplicates(subset=["category"], keep="first")
    _joint_cdi_sem_cache = dict(zip(sem_df["category"], sem_df["cdi_semantic"]))
    return _joint_cdi_sem_cache


def _joint_base_rgb_for_category_name(cat_name):
    import matplotlib.colors as mcolors
    k = str(cat_name).strip().lower()
    sem_map = _joint_load_category_to_cdi_semantic()
    sem = sem_map.get(k, "other")
    if sem == "animals" and k in _JOINT_ANIMAL_PAIR_HEX:
        hx = _JOINT_ANIMAL_PAIR_HEX[k]
    else:
        hx = _CDI_SEMANTIC_COLORS_JOINT.get(sem, _CDI_SEMANTIC_COLORS_JOINT["other"])
    return np.array(mcolors.to_rgb(hx), dtype=np.float64)


def _tsne_bold_ticklabels(ax):
    for lb in list(ax.get_xticklabels()) + list(ax.get_yticklabels()):
        lb.set_fontweight("bold")


def _tsne_style_axes(ax, title, xlabel, ylabel, title_fs=None):
    fs_t = title_fs if title_fs is not None else _TSNE_TITLE_FS
    ax.set_title(title, fontsize=fs_t, fontweight="bold", pad=10)
    ax.set_xlabel(xlabel, fontsize=_TSNE_AXIS_FS, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=_TSNE_AXIS_FS, fontweight="bold")
    ax.tick_params(axis="both", which="major", labelsize=_TSNE_TICK_FS, width=1.3, length=6)
    _tsne_bold_ticklabels(ax)


def _tsne_style_colorbar(cbar, label):
    cbar.set_label(label, fontsize=_TSNE_CBAR_LABEL_FS, fontweight="bold")
    cbar.ax.tick_params(labelsize=_TSNE_CBAR_TICK_FS)
    for lb in cbar.ax.get_yticklabels():
        lb.set_fontweight("bold")




def _tsne_colorbar(ax, mappable, label):
    """Colorbar with the same height as the main axes (right side)."""
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4.8%", pad=0.12)
    cbar = ax.figure.colorbar(mappable, cax=cax)
    _tsne_style_colorbar(cbar, label)
    return cbar
def plot_tsne_one_category(
    category,
    bv_X,
    bv_centroid,
    dist_bv_to_bv,
    out_path,
    perplexity=30,
    random_state=42,
    zscore=True,
    title_suffix="",
    model_label="CLIP",
):
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from matplotlib.collections import PathCollection
    from sklearn.manifold import TSNE

    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42

    n_bv = bv_X.shape[0]
    X_all = np.vstack([bv_X, bv_centroid.reshape(1, -1)])

    if zscore:
        mean, std = X_all.mean(axis=0), X_all.std(axis=0)
        std = np.where(std > 1e-8, std, 1.0)
        X_all = (X_all - mean) / std

    n_all = X_all.shape[0]
    perplexity = min(perplexity, max(2, (n_all - 1) // 3))
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=random_state,
        init="pca",
        learning_rate="auto",
        max_iter=1500,
    )
    coords = tsne.fit_transform(X_all)

    bx, by = coords[:n_bv].T
    bv_cx, bv_cy = coords[n_bv]

    plt.style.use("seaborn-v0_8-white")
    fig, ax = plt.subplots(figsize=(9.2, 7.2), facecolor="#fbfbfc")

    sc = ax.scatter(
        bx,
        by,
        c=dist_bv_to_bv,
        s=58,
        cmap="magma",
        alpha=0.95,
        edgecolors="white",
        linewidths=0.45,
        label="BV exemplars",
        zorder=3,
        rasterized=False,
    )
    _tsne_colorbar(ax, sc, "Distance to BV centroid")

    ax.scatter([bv_cx], [bv_cy], s=_JOINT_CENTROID_S_FILLED, marker="o", c="#1958d1", edgecolors="white", linewidths=1.5, label="BV centroid", zorder=6, rasterized=False)
    ax.scatter([bv_cx], [bv_cy], s=_JOINT_CENTROID_S_RING, marker="o", facecolors="none", edgecolors="#1958d1", linewidths=1.5, alpha=0.45, zorder=5, rasterized=False)

    _tsne_style_axes(
        ax,
        f"{category}{title_suffix} | t-SNE ({model_label.upper()})",
        "t-SNE 1",
        "t-SNE 2",
    )
    legend = ax.legend(loc="best", frameon=True, markerscale=1.0, prop={"weight": "bold", "size": _TSNE_LEGEND_FS})
    for h in legend.legend_handles:
        if isinstance(h, PathCollection):
            h.set_sizes([58])
    ax.set_aspect("equal")
    ax.grid(False)
    ax.set_axisbelow(True)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    pdf_path = out_path.with_suffix(".pdf")
    plt.tight_layout()
    plt.savefig(out_path, dpi=190, facecolor=fig.get_facecolor())
    plt.savefig(pdf_path, format="pdf")
    plt.close()
    print(f"  Saved t-SNE: {out_path}")
    print(f"  Saved t-SNE PDF: {pdf_path}")


def _joint_tsne_fit(category_specs, perplexity=30, random_state=42, zscore=True, background_bundles=None):
    """Joint 2D layout for highlighted categories.

    If ``background_bundles`` is None (highlight_only): stack each category's exemplars plus its BV centroid row,
    z-score, single t-SNE (original behavior).

    If ``background_bundles`` is a list of ``{name, bv_X}`` for all categories to include in the fit: t-SNE runs on
    **exemplars only** (no centroid rows). Returned centroid positions are the **mean** of each highlighted
    category's exemplar coordinates in 2D (so the star sits at the t-SNE barycenter of that cloud).
    """
    from sklearn.manifold import TSNE

    names = [s["name"] for s in category_specs]
    if len(names) < 2 or len(names) > 4:
        raise ValueError(f"joint t-SNE expects 2–4 categories, got {len(names)}")

    if background_bundles is not None:
        bg_by_name = {b["name"]: np.asarray(b["bv_X"], dtype=np.float64) for b in background_bundles}
        for s in category_specs:
            if s["name"] not in bg_by_name:
                raise ValueError(f"joint highlight {s['name']!r} missing from background_bundles")
            Xh = np.asarray(s["bv_X"], dtype=np.float64)
            Xb = bg_by_name[s["name"]]
            if Xh.shape != Xb.shape or not np.allclose(Xh, Xb):
                raise ValueError(
                    f"background bv_X for {s['name']!r} must match highlight spec (same filtered exemplars)"
                )
        ordered_names = sorted(bg_by_name.keys(), key=lambda x: str(x).lower())
        X_all = np.vstack([bg_by_name[nm] for nm in ordered_names])
        if zscore:
            mean, std = X_all.mean(axis=0), X_all.std(axis=0)
            std = np.where(std > 1e-8, std, 1.0)
            X_all = (X_all - mean) / std
        n_all = X_all.shape[0]
        perplexity = min(float(perplexity), max(2.0, (n_all - 1) // 3))
        tsne = TSNE(
            n_components=2,
            perplexity=perplexity,
            random_state=random_state,
            init="pca",
            learning_rate="auto",
            max_iter=1500,
        )
        coords = tsne.fit_transform(X_all)
        row = 0
        coords_by_name = {}
        for nm in ordered_names:
            n_i = bg_by_name[nm].shape[0]
            coords_by_name[nm] = coords[row : row + n_i]
            row += n_i
        exemplar_coords = [coords_by_name[s["name"]] for s in category_specs]
        centroid_coords = [np.mean(ec, axis=0) for ec in exemplar_coords]
        return {"exemplar_coords": exemplar_coords, "centroid_coords": centroid_coords, "names": names}

    blocks, n_per = [], []
    for s in category_specs:
        X = np.asarray(s["bv_X"], dtype=np.float64)
        cen = np.asarray(s["bv_centroid"], dtype=np.float64).reshape(1, -1)
        blocks.append(X)
        blocks.append(cen)
        n_per.append(X.shape[0])
    X_all = np.vstack(blocks)
    if zscore:
        mean, std = X_all.mean(axis=0), X_all.std(axis=0)
        std = np.where(std > 1e-8, std, 1.0)
        X_all = (X_all - mean) / std
    n_all = X_all.shape[0]
    perplexity = min(float(perplexity), max(2.0, (n_all - 1) // 3))
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=random_state,
        init="pca",
        learning_rate="auto",
        max_iter=1500,
    )
    coords = tsne.fit_transform(X_all)
    exemplar_coords, centroid_coords, row = [], [], 0
    for n_i in n_per:
        exemplar_coords.append(coords[row : row + n_i])
        row += n_i
        centroid_coords.append(coords[row])
        row += 1
    return {"exemplar_coords": exemplar_coords, "centroid_coords": centroid_coords, "names": names}


def _joint_tsne_distance_range(category_specs):
    all_d = np.concatenate([np.asarray(s["dist_bv_to_bv"], dtype=np.float64) for s in category_specs])
    d_min, d_max = float(all_d.min()), float(all_d.max())
    if d_max - d_min < 1e-12:
        d_max = d_min + 1.0
    return d_min, d_max


def _joint_category_label_offsets(n_cats, span):
    """Data-space (dx, dy) for category text relative to each centroid."""
    d = max(span * 0.075, 1e-6)
    if n_cats <= 1:
        return [(d, d)]
    if n_cats == 2:
        return [(d, d * 0.9), (-d * 1.05, -d * 0.85)]
    if n_cats == 3:
        return [(d * 1.05, d * 0.35), (-d * 0.85, d * 1.05), (d * 0.15, -d * 1.12)]
    # 4 categories: push labels toward quadrants
    return [
        (d * 1.05, d * 0.75),
        (-d * 1.1, d * 0.75),
        (-d * 1.05, -d * 0.8),
        (d * 1.0, -d * 0.85),
    ]


def _plot_joint_tsne_on_ax(
    ax,
    category_specs,
    layout,
    d_min,
    d_max,
    *,
    model_label,
    title_suffix="",
    show_cbar=True,
    colors=None,
    color_mode="magma",
):
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    from matplotlib import cm
    from matplotlib.colors import Normalize
    from matplotlib.lines import Line2D

    names = layout["names"]
    exemplar_coords = layout["exemplar_coords"]
    centroid_coords = layout["centroid_coords"]
    n_names = len(names)
    cbar_drawn = False
    cen_col = "#1958d1"

    def _gray_cmap():
        import matplotlib as _mpl
        try:
            return _mpl.colormaps["gray"]
        except (AttributeError, KeyError, TypeError):
            return cm.get_cmap("gray")

    if color_mode == "hue_saturation":
        # Plot B/C CDI semantic base RGB + gray blend by distance (saturation / vividness).
        all_d = np.concatenate([np.asarray(spec["dist_bv_to_bv"], dtype=np.float64) for spec in category_specs])
        p_lo, p_hi = np.percentile(all_d, [6.0, 94.0])
        if p_hi - p_lo < 1e-12:
            p_lo, p_hi = float(all_d.min()), float(all_d.max()) + 1e-6
        gam = float(globals().get("tsne_joint_sat_gamma", 0.42))
        s_floor = float(globals().get("tsne_joint_sat_floor", 0.12))
        gray = np.array([0.90, 0.90, 0.91], dtype=np.float64)
        legend_face = []

        for i, spec in enumerate(category_specs):
            ex = exemplar_coords[i]
            d = np.asarray(spec["dist_bv_to_bv"], dtype=np.float64)
            t = np.clip((d - p_lo) / (p_hi - p_lo + 1e-12), 0, 1)
            wmix = s_floor + (1.0 - s_floor) * (t ** gam)
            base = _joint_base_rgb_for_category_name(names[i])
            rgb = (1.0 - wmix)[:, np.newaxis] * gray + wmix[:, np.newaxis] * base
            rgb = np.clip(rgb, 0, 1)
            legend_face.append(tuple(base))
            ax.scatter(
                ex[:, 0],
                ex[:, 1],
                c=rgb,
                s=58,
                alpha=0.95,
                edgecolors="white",
                linewidths=0.45,
                zorder=3,
                rasterized=False,
            )
            if show_cbar and not cbar_drawn:
                sm = cm.ScalarMappable(norm=Normalize(vmin=p_lo, vmax=p_hi), cmap=_gray_cmap())
                sm.set_array([])
                _tsne_colorbar(ax, sm, "Distance to own BV centroid (vividness)")
                cbar_drawn = True
            cx, cy = centroid_coords[i]
            ax.scatter(
                [cx],
                [cy],
                s=_JOINT_CENTROID_S_FILLED,
                marker="o",
                c=cen_col,
                edgecolors="white",
                linewidths=1.5,
                zorder=6,
                rasterized=False,
            )
            ax.scatter(
                [cx],
                [cy],
                s=_JOINT_CENTROID_S_RING,
                marker="o",
                facecolors="none",
                edgecolors=cen_col,
                linewidths=1.5,
                alpha=0.45,
                zorder=5,
                rasterized=False,
            )

        legend_elems = [
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=legend_face[j],
                markersize=8,
                markeredgecolor="white",
                markeredgewidth=0.45,
                label=f"{names[j]} (CDI base)",
            )
            for j in range(n_names)
        ]
        legend_elems.append(
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=cen_col,
                markersize=10,
                markeredgecolor="white",
                markeredgewidth=0.8,
                label="BV centroid + category label",
            )
        )
    else:
        for i, spec in enumerate(category_specs):
            ex = exemplar_coords[i]
            d = np.asarray(spec["dist_bv_to_bv"], dtype=np.float64)
            sc = ax.scatter(
                ex[:, 0],
                ex[:, 1],
                c=d,
                cmap="magma",
                vmin=d_min,
                vmax=d_max,
                s=58,
                alpha=0.95,
                edgecolors="white",
                linewidths=0.45,
                zorder=3,
                rasterized=False,
            )
            if show_cbar and not cbar_drawn:
                _tsne_colorbar(ax, sc, "Distance to own BV centroid")
                cbar_drawn = True
            cx, cy = centroid_coords[i]
            ax.scatter(
                [cx],
                [cy],
                s=_JOINT_CENTROID_S_FILLED,
                marker="o",
                c=cen_col,
                edgecolors="white",
                linewidths=1.5,
                zorder=6,
                rasterized=False,
            )
            ax.scatter(
                [cx],
                [cy],
                s=_JOINT_CENTROID_S_RING,
                marker="o",
                facecolors="none",
                edgecolors=cen_col,
                linewidths=1.5,
                alpha=0.45,
                zorder=5,
                rasterized=False,
            )

        legend_elems = [
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor="#6c3483",
                markersize=8,
                markeredgecolor="white",
                markeredgewidth=0.45,
                label="exemplars (magma = dist. to own BV centroid)",
            ),
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=cen_col,
                markersize=10,
                markeredgecolor="white",
                markeredgewidth=0.8,
                label="BV centroid + category label",
            ),
        ]

    cat_str = " + ".join(names)
    _tsne_style_axes(
        ax,
        f"{cat_str}{title_suffix} | joint t-SNE ({model_label.upper()})",
        "t-SNE 1",
        "t-SNE 2",
    )
    ax.set_aspect("equal")
    ax.grid(False)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    span = max(x1 - x0, y1 - y0)
    offs = _joint_category_label_offsets(n_names, span)
    for i in range(n_names):
        cx, cy = centroid_coords[i]
        dx, dy = offs[i % len(offs)]
        ax.annotate(
            names[i],
            xy=(float(cx), float(cy)),
            xytext=(float(cx + dx), float(cy + dy)),
            textcoords="data",
            ha="center",
            va="center",
            fontsize=_TSNE_LEGEND_FS,
            fontweight="bold",
            color="0.15",
            arrowprops=dict(arrowstyle="-", color="0.35", lw=1.25, shrinkA=5, shrinkB=3),
            zorder=8,
        )

    ax.legend(handles=legend_elems, loc="best", frameon=True, prop={"weight": "bold", "size": _TSNE_LEGEND_FS})



def plot_tsne_multi_categories(
    category_specs,
    out_path,
    perplexity=30,
    random_state=42,
    zscore=True,
    title_suffix="",
    model_label="CLIP",
    background_bundles=None,
    joint_color_mode="magma",
):
    """Joint figure for 2–4 categories; optional global t-SNE via ``background_bundles``."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    _fs_joint_single = globals().get("_TSNE_JOINT_SINGLE_FIGSIZE", (9.0, 6.6))
    d_min, d_max = _joint_tsne_distance_range(category_specs)
    layout = _joint_tsne_fit(
        category_specs,
        perplexity=perplexity,
        random_state=random_state,
        zscore=zscore,
        background_bundles=background_bundles,
    )
    plt.style.use("seaborn-v0_8-white")
    fig, ax = plt.subplots(figsize=_fs_joint_single, facecolor="#fbfbfc")
    _plot_joint_tsne_on_ax(
        ax,
        category_specs,
        layout,
        d_min,
        d_max,
        model_label=model_label,
        title_suffix=title_suffix,
        show_cbar=True,
        color_mode=joint_color_mode,
    )
    pdf_path = out_path.with_suffix(".pdf")
    plt.tight_layout()
    plt.savefig(out_path, dpi=190, facecolor=fig.get_facecolor())
    plt.savefig(pdf_path, format="pdf")
    plt.close()
    print(f"  Saved joint t-SNE: {out_path}")
    print(f"  Saved joint t-SNE PDF: {pdf_path}")


def plot_tsne_multi_categories_side_by_side(
    specs_clip,
    specs_dinov3,
    out_path,
    perplexity=30,
    random_state=42,
    zscore=True,
    title_suffix="",
    background_bundles_clip=None,
    background_bundles_dinov3=None,
    joint_color_mode="magma",
):
    """CLIP vs DINOv3 joint multi-category t-SNE; shared color scale for distance across panels."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    mpl.rcParams["pdf.fonttype"] = 42
    mpl.rcParams["ps.fonttype"] = 42
    _fs_joint_sbs = globals().get("_TSNE_JOINT_CLIP_DINO_FIGSIZE", (18.0, 6.6))
    parts = []
    for s in specs_clip:
        parts.append(np.asarray(s["dist_bv_to_bv"], dtype=np.float64))
    for s in specs_dinov3:
        parts.append(np.asarray(s["dist_bv_to_bv"], dtype=np.float64))
    all_d = np.concatenate(parts)
    d_min, d_max = float(all_d.min()), float(all_d.max())
    if d_max - d_min < 1e-12:
        d_max = d_min + 1.0
    layout_c = _joint_tsne_fit(
        specs_clip,
        perplexity=perplexity,
        random_state=random_state,
        zscore=zscore,
        background_bundles=background_bundles_clip,
    )
    layout_d = _joint_tsne_fit(
        specs_dinov3,
        perplexity=perplexity,
        random_state=random_state,
        zscore=zscore,
        background_bundles=background_bundles_dinov3,
    )
    plt.style.use("seaborn-v0_8-white")
    names = layout_c["names"]
    slug_title = " + ".join(names)

    fig, axes = plt.subplots(1, 2, figsize=_fs_joint_sbs, facecolor="#fbfbfc")
    _plot_joint_tsne_on_ax(
        axes[0],
        specs_clip,
        layout_c,
        d_min,
        d_max,
        model_label="CLIP",
        title_suffix=title_suffix,
        show_cbar=True,
        color_mode=joint_color_mode,
    )
    _plot_joint_tsne_on_ax(
        axes[1],
        specs_dinov3,
        layout_d,
        d_min,
        d_max,
        model_label="DINOv3",
        title_suffix=title_suffix,
        show_cbar=True,
        color_mode=joint_color_mode,
    )
    fig.suptitle(f"{slug_title} | CLIP vs DINOv3 joint t-SNE{title_suffix}", fontsize=16, fontweight="bold", y=1.02)
    pdf_path = out_path.with_suffix(".pdf")
    fig.tight_layout()
    fig.savefig(out_path, dpi=190, facecolor=fig.get_facecolor(), bbox_inches="tight")
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved joint side-by-side t-SNE: {out_path}")
    print(f"  Saved joint side-by-side t-SNE PDF: {pdf_path}")

    def _collect_xy(layout):
        xs, ys = [], []
        for row in layout["exemplar_coords"]:
            xs.extend(row[:, 0].tolist())
            ys.extend(row[:, 1].tolist())
        for pt in layout["centroid_coords"]:
            xs.append(float(pt[0]))
            ys.append(float(pt[1]))
        return np.array(xs), np.array(ys)

    xc, yc = _collect_xy(layout_c)
    xd, yd = _collect_xy(layout_d)
    all_x = np.concatenate([xc, xd])
    all_y = np.concatenate([yc, yd])
    x_min, x_max = float(all_x.min()), float(all_x.max())
    y_min, y_max = float(all_y.min()), float(all_y.max())
    x_pad = max((x_max - x_min) * 0.08, 1e-6)
    y_pad = max((y_max - y_min) * 0.08, 1e-6)

    fig2, axes2 = plt.subplots(1, 2, figsize=_fs_joint_sbs, facecolor="#fbfbfc")
    _plot_joint_tsne_on_ax(
        axes2[0],
        specs_clip,
        layout_c,
        d_min,
        d_max,
        model_label="CLIP",
        title_suffix=title_suffix,
        show_cbar=True,
        color_mode=joint_color_mode,
    )
    _plot_joint_tsne_on_ax(
        axes2[1],
        specs_dinov3,
        layout_d,
        d_min,
        d_max,
        model_label="DINOv3",
        title_suffix=title_suffix,
        show_cbar=True,
        color_mode=joint_color_mode,
    )
    for ax in axes2:
        ax.set_xlim(x_min - x_pad, x_max + x_pad)
        ax.set_ylim(y_min - y_pad, y_max + y_pad)
    fig2.suptitle(f"{slug_title} | CLIP vs DINOv3 joint t-SNE (aligned axes){title_suffix}", fontsize=16, fontweight="bold", y=1.02)
    out_aligned = out_path.parent / (out_path.stem + "_aligned" + out_path.suffix)
    pdf_aligned = out_aligned.with_suffix(".pdf")
    fig2.tight_layout()
    fig2.savefig(out_aligned, dpi=190, facecolor=fig2.get_facecolor(), bbox_inches="tight")
    fig2.savefig(pdf_aligned, format="pdf", bbox_inches="tight")
    plt.close(fig2)
    print(f"  Saved joint side-by-side aligned t-SNE: {out_aligned}")
    print(f"  Saved joint side-by-side aligned t-SNE PDF: {pdf_aligned}")


## RDM helpers (BV vs THINGS joint matrices)

These functions support an optional RDM path when `rdm_top12_and_things` is enabled. The **default pipeline does not load THINGS**; leave RDM off for the BV-only centroid + t-SNE workflow.

In [7]:
def plot_rdm_one_category(category, bv_X, things_X, out_path, title_suffix="", max_things=None, random_state=42):
    """Plot RDM (1 - Pearson correlation) for [BV exemplars | THINGS exemplars]."""
    import matplotlib.pyplot as plt

    n_bv, n_things = bv_X.shape[0], things_X.shape[0]
    if max_things is not None and n_things > max_things:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(n_things, size=max_things, replace=False)
        things_X = things_X[idx]
        n_things = max_things
    X = np.vstack([bv_X, things_X])
    n = X.shape[0]
    corr = np.corrcoef(X)
    rdm = 1.0 - np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=2.0)
    np.fill_diagonal(rdm, 0)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(rdm, cmap="viridis", aspect="equal", vmin=0, vmax=2)
    ax.axvline(n_bv - 0.5, color="white", linewidth=1.5)
    ax.axhline(n_bv - 0.5, color="white", linewidth=1.5)
    ax.set_xticks([n_bv // 2 - 0.5, n_bv + (n_things - 1) // 2])
    ax.set_xticklabels(["BV (top-12)", "THINGS"])
    ax.set_yticks([n_bv // 2 - 0.5, n_bv + (n_things - 1) // 2])
    ax.set_yticklabels(["BV (top-12)", "THINGS"])
    ax.set_title(f"RDM: {category}{title_suffix} — 1 - correlation (BV vs THINGS)")
    plt.colorbar(im, ax=ax, label="Dissimilarity (1 - r)")
    pdf_path = out_path.with_suffix(".pdf")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.close()
    print(f"  Saved RDM: {out_path}")
    print(f"  Saved RDM PDF: {pdf_path}")

In [8]:
def plot_rdm_all_categories(bv_means, th_means, category_names, out_path):
    """Plot one RDM: rows/cols = [BV_avg_cat1, ..., BV_avg_catN, THINGS_avg_cat1, ..., THINGS_avg_catN].
    bv_means, th_means: (N, dim). Dissimilarity = 1 - Pearson r."""
    import matplotlib.pyplot as plt

    N = len(category_names)
    X = np.vstack([np.asarray(bv_means), np.asarray(th_means)])
    corr = np.corrcoef(X)
    rdm = 1.0 - np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=2.0)
    np.fill_diagonal(rdm, 0)

    fig, ax = plt.subplots(figsize=(max(8, N * 0.5), max(7, N * 0.45)))
    im = ax.imshow(rdm, cmap="viridis", aspect="equal", vmin=0, vmax=2)
    ax.axvline(N - 0.5, color="white", linewidth=1.5)
    ax.axhline(N - 0.5, color="white", linewidth=1.5)
    ticks = list(range(N)) + list(range(N, 2 * N))
    labels = [f"{c} (BV)" for c in category_names] + [f"{c} (TH)" for c in category_names]
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(labels, rotation=90, ha="right", fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_title("RDM: all selected categories — BV avg (top-12) vs THINGS avg (1 - r)")
    plt.colorbar(im, ax=ax, label="Dissimilarity (1 - r)")
    pdf_path = out_path.with_suffix(".pdf")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.close()
    print(f"  Saved RDM (all categories): {out_path}")
    print(f"  Saved RDM PDF (all categories): {pdf_path}")

## Run pipeline: load BV embeddings, compute BV→BV-centroid distances, save CSVs

In [9]:
out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

allowed = load_allowed_categories()
allowed_sorted = sorted(allowed) if allowed else None
print(f"Target categories from included list: {len(allowed_sorted) if allowed_sorted else 'all'}")

results_by_embedding = {}
plotA_selected_categories = load_plotA_selected_categories(plotA_selected_csv) if tsne_use_plotA_selection else []
valid_file_stems = load_valid_file_stems(PER_FILE_PRECISION_CSV, threshold=PRECISION_THRESHOLD)
plotA_sampled_stems = load_sampled_high_confidence_stems(
    plotA_sampled_exemplars_csv, plotA_selected_categories, n_exemplars=plotA_n_exemplars,
    valid_file_stems=valid_file_stems,
) if tsne_use_plotA_exemplars else {}
for embedding in embeddings_to_run:
    prefix = f"{output_metric_slug}_{embedding}"
    print(f"\n=== Running embedding: {embedding} ===")
    print("Loading BV embeddings (from clip-filtered metadata; original per-image embeddings)...")
    bv_embeddings, bv_ids = load_bv_embeddings_from_filtered(
        embedding, metadata_csv, allowed, EXCLUDED_SUBJECT, MIN_EXEMPLARS_BV
    )

    if allowed_sorted is not None:
        categories_common = [c for c in allowed_sorted if c in bv_embeddings]
    else:
        categories_common = sorted(set(bv_embeddings.keys()))

    print(f"Categories with BV embeddings: {len(categories_common)}")
    if allowed_sorted is not None and len(categories_common) != len(allowed_sorted):
        missing = [c for c in allowed_sorted if c not in categories_common]
        print(f"Missing categories ({len(missing)}): {missing[:20]}{' ...' if len(missing) > 20 else ''}")

    summary_rows, exemplar_rows = run_distances(
        bv_embeddings, bv_ids, categories_common
    )
    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.sort_values("mean_bv_to_bv_centroid", ascending=False).reset_index(drop=True)
    summary_path = out_dir / f"{prefix}_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Saved summary: {summary_path}")

    if save_exemplar_csv:
        exemplar_df = pd.DataFrame(exemplar_rows)
        exemplar_path = out_dir / f"{prefix}_per_exemplar.csv"
        exemplar_df.to_csv(exemplar_path, index=False)
        print(f"Saved per-exemplar: {exemplar_path}")
    else:
        exemplar_df = pd.DataFrame(exemplar_rows)

    results_by_embedding[embedding] = {
        "prefix": prefix,
        "summary_df": summary_df,
        "exemplar_df": exemplar_df,
        "categories_common": categories_common,
        "bv_embeddings": bv_embeddings,
        "bv_ids": bv_ids,
    }

# preserve canonical variables for downstream plotting cells
embedding = embeddings_to_run[0]
prefix = results_by_embedding[embedding]["prefix"]
summary_df = results_by_embedding[embedding]["summary_df"]
exemplar_df = results_by_embedding[embedding]["exemplar_df"]
categories_common = results_by_embedding[embedding]["categories_common"]
bv_embeddings = results_by_embedding[embedding]["bv_embeddings"]
print(f"\nDefault downstream view set to: {embedding}")

Target categories from included list: 85

=== Running embedding: dinov3 ===
Loading BV embeddings (from clip-filtered metadata; original per-image embeddings)...


/tmp/ipykernel_2432625/1483170638.py:110: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cat_to_rows = meta.groupby("class_name", sort=False).apply(


Categories with BV embeddings: 85


Categories: 100%|██████████| 85/85 [00:08<00:00, 10.04it/s]


Saved summary: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_summary.csv
Saved per-exemplar: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_per_exemplar.csv

Default downstream view set to: dinov3


## Histogram + ranked barplot (category-level BV→BV centroid spread)

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# Scatter (existing diagnostic)
# -----------------------------
fig, ax = plt.subplots(figsize=(7, 6))
ax.hist(summary_df["mean_bv_to_bv_centroid"], bins=24, alpha=0.85)
ax.set_xlabel("Mean BV exemplar distance to BV centroid (within-category spread)")
ax.set_ylabel("Category count")
ax.set_title(f"Distribution of category spread to BV centroid ({embedding})")
plt.tight_layout()
scatter_path = out_dir / f"{prefix}_spread_hist_bv_centroid.png"
scatter_pdf_path = scatter_path.with_suffix(".pdf")
plt.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.savefig(scatter_pdf_path, bbox_inches="tight")
plt.close()
print(f"Saved scatter: {scatter_path}")
print(f"Saved scatter PDF: {scatter_pdf_path}")

# -------------------------------------------------------------
# Ranked barplot: category-wise centroid distance (high -> low)
# color-coded by CDI semantic categories
# -------------------------------------------------------------
semantic_csv = project_root / "data/long_tailed_dist_prop_included_categories.csv"
sem_df = pd.read_csv(semantic_csv, usecols=["category", "cdi_semantic"])\
    .dropna(subset=["category", "cdi_semantic"])\
    .drop_duplicates(subset=["category"], keep="first")
sem_df["category"] = sem_df["category"].astype(str).str.strip().str.lower()
sem_df["cdi_semantic"] = sem_df["cdi_semantic"].astype(str).str.strip().str.lower()

bar_df = summary_df[["category", "mean_bv_to_bv_centroid"]].copy()
bar_df = bar_df.merge(sem_df, on="category", how="left")
bar_df["cdi_semantic"] = bar_df["cdi_semantic"].fillna("unknown")
bar_df = bar_df.sort_values("mean_bv_to_bv_centroid", ascending=False).reset_index(drop=True)

CDI_SEMANTIC_ORDER = [
    "animals", "body_parts", "clothing", "food_drink", "furniture_rooms",
    "household", "outside", "people", "toys", "vehicles", "other",
]
CDI_SEMANTIC_COLORS = {
    "animals": "#4DB8A8",
    "body_parts": "#E87A5F",
    "clothing": "#9B7EC8",
    "food_drink": "#E8A54C",
    "furniture_rooms": "#6BAB7A",
    "household": "#D97B9E",
    "outside": "#5B9BD5",
    "people": "#E8C44C",
    "toys": "#B07CC8",
    "vehicles": "#6BA3D5",
    "other": "#8B9A9E",
}

semantic_order = [s for s in CDI_SEMANTIC_ORDER if s in set(bar_df["cdi_semantic"])]
if len(semantic_order) == 0:
    semantic_order = sorted(bar_df["cdi_semantic"].unique())
semantic_to_color = {s: CDI_SEMANTIC_COLORS.get(s, CDI_SEMANTIC_COLORS["other"]) for s in semantic_order}
bar_colors = [semantic_to_color.get(s, CDI_SEMANTIC_COLORS["other"]) for s in bar_df["cdi_semantic"]]

fig_w = max(18, 0.22 * len(bar_df))
fig, ax = plt.subplots(figsize=(fig_w, 6.8), facecolor="#fbfbfc")
ax.bar(bar_df["category"], bar_df["mean_bv_to_bv_centroid"], color=bar_colors, edgecolor="none", linewidth=0)
ax.set_title("Category-wise mean distance to BV centroid (ranked high to low)", fontsize=13, pad=10)
ax.set_xlabel("Category")
ax.set_ylabel("Mean BV exemplar distance to BV centroid")
ax.tick_params(axis="x", rotation=90, labelsize=8)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)

handles = [
    plt.Line2D([0], [0], marker="s", color="none", markerfacecolor=semantic_to_color[s], markersize=9, label=s)
    for s in semantic_order
]
ax.legend(handles=handles, title="CDI semantic", ncol=min(5, len(handles)), frameon=True, loc="upper right")

plt.tight_layout()
barplot_path = out_dir / f"{prefix}_category_ranked_bv_centroid_distance_by_semantic.png"
barplot_pdf_path = barplot_path.with_suffix(".pdf")
plt.savefig(barplot_path, dpi=180, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.savefig(barplot_pdf_path, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"Saved ranked semantic barplot: {barplot_path}")
print(f"Saved ranked semantic barplot PDF: {barplot_pdf_path}")

Saved scatter: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_spread_hist_bv_centroid.png
Saved scatter PDF: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_spread_hist_bv_centroid.pdf
Saved ranked semantic barplot: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_category_ranked_bv_centroid_distance_by_semantic.png
Saved ranked semantic barplot PDF: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_category_ranked_bv_centroid_distance_by_semantic.pdf


## Optional: save per-exemplar CSV (`dist_to_bv_centroid` only)

In [11]:
if save_exemplar_csv:
    exemplar_df = pd.DataFrame(exemplar_rows)
    exemplar_path = out_dir / f"{prefix}_per_exemplar.csv"
    exemplar_df.to_csv(exemplar_path, index=False)
    print(f"Saved per-exemplar: {exemplar_path}")

Saved per-exemplar: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_per_exemplar.csv


## Optional: t-SNE for one or all categories (BV-only manifold)

In [12]:
if tsne_all_categories or tsne_category or tsne_select_top_bottom_by_distance or tsne_one_per_semantic:
    for embedding in embeddings_to_run:
        if embedding not in results_by_embedding:
            print(f"Skipping t-SNE for {embedding}: no loaded results")
            continue

        prefix = results_by_embedding[embedding]["prefix"]
        summary_df = results_by_embedding[embedding]["summary_df"]
        categories_common = results_by_embedding[embedding]["categories_common"]
        bv_embeddings = results_by_embedding[embedding]["bv_embeddings"]
        bv_ids = results_by_embedding[embedding]["bv_ids"]

        tsne_dir = out_dir / f"{prefix}_tsne_by_category"
        tsne_dir.mkdir(exist_ok=True)

        rdm_dir = out_dir / f"{prefix}_rdm_by_category" if rdm_top12_and_things else None
        if rdm_dir is not None:
            rdm_dir.mkdir(exist_ok=True)

        if tsne_all_categories:
            categories_to_plot = categories_common
            print(f"[{embedding}] Running t-SNE for all {len(categories_to_plot)} categories...")
        else:
            if tsne_use_plotA_selection and plotA_selected_categories:
                requested = list(plotA_selected_categories)
                category_to_rank_group = {str(c).strip().lower(): "plotA" for c in requested}
                print(f"[{embedding}] Using Plot A selected categories: {requested}")
            elif tsne_select_top_bottom_by_distance:
                rank_col = tsne_rank_metric
                if rank_col not in summary_df.columns:
                    raise ValueError(f"Ranking metric '{rank_col}' not found in summary_df columns: {list(summary_df.columns)}")

                ranked = summary_df.sort_values(rank_col, ascending=False).reset_index(drop=True)
                top_df = ranked.head(tsne_top_k).copy()
                top_df["rank_group"] = "top"
                bottom_df = ranked.tail(tsne_bottom_k).copy()
                bottom_df["rank_group"] = "bottom"

                selected = pd.concat([top_df, bottom_df], axis=0)
                selected = selected.drop_duplicates(subset=["category"], keep="first")
                requested = selected["category"].tolist()
                category_to_rank_group = dict(zip(selected["category"], selected["rank_group"]))

                print(f"[{embedding}] Selected top {tsne_top_k} + bottom {tsne_bottom_k} by {rank_col}:")
                print(selected[["category", rank_col, "rank_group"]].to_string(index=False))

            elif tsne_one_per_semantic:
                selected_df = select_one_category_per_semantic(tsne_semantic_source_csv)
                requested = selected_df["category"].tolist()
                category_to_rank_group = {c: "selected" for c in requested}
                print(f"[{embedding}] One category selected per CDI semantic group:")
                print(selected_df.to_string(index=False))
            else:
                requested = [tsne_category] if isinstance(tsne_category, str) else list(tsne_category)
                category_to_rank_group = {str(c).strip().lower(): "manual" for c in requested}

            categories_to_plot = []
            for cat in requested:
                c = str(cat).strip().lower()
                match = next((x for x in categories_common if x.lower() == c), None)
                if match:
                    categories_to_plot.append(match)

            print(f"[{embedding}] Running t-SNE for {len(categories_to_plot)} categories: {categories_to_plot}")
            if len(categories_to_plot) == 0:
                raise ValueError(f"[{embedding}] No categories selected for t-SNE. Check ranking metric and upstream summary_df generation.")

        print(f"[{embedding}] t-SNE output directory: {tsne_dir}")

        # Optional similarity ranking support; t-SNE now uses all valid exemplars per category.
        tsne_ranked_stems = {}
        if tsne_top12_by_similarity and embedding == "clip" and SIMILARITY_CSV.exists():
            sim_df = pd.read_csv(SIMILARITY_CSV)
            sim_col = next((c for c in ["cosine_similarity", "similarity", "sim", "cos_sim"] if c in sim_df.columns), None)
            cat_col = next((c for c in ["class_name", "category", "cat"] if c in sim_df.columns), None)
            emb_col = next((c for c in ["original_embedding_name", "embedding_name", "embedding_id", "embedding"] if c in sim_df.columns), None)
            if "frame_path" in sim_df.columns and (cat_col is None or emb_col is None):
                sim_df["embedding_name"] = sim_df["frame_path"].astype(str).str.rsplit("/", n=1).str[-1]
                sim_df["embedding_name"] = sim_df["embedding_name"].apply(lambda x: x if str(x).endswith(".npy") else f"{x}.npy")
                sim_df["class_name"] = sim_df["frame_path"].astype(str).str.rsplit("/", n=2).str[-2].str.strip().str.lower()
            if sim_col is not None:
                sim_df["sim_value"] = pd.to_numeric(sim_df[sim_col], errors="coerce")
                if "class_name" not in sim_df.columns and cat_col:
                    sim_df["class_name"] = sim_df[cat_col].astype(str).str.strip().str.lower()
                if "embedding_name" not in sim_df.columns and emb_col:
                    sim_df["embedding_name"] = sim_df[emb_col].astype(str).str.strip()
                    sim_df["embedding_name"] = sim_df["embedding_name"].apply(lambda x: x if x.endswith(".npy") else f"{x}.npy")
                meta = load_filtered_metadata(metadata_csv, excluded_subject=EXCLUDED_SUBJECT)
                meta["original_embedding_name"] = meta["original_embedding_name"].astype(str).str.strip()
                meta["original_embedding_name"] = meta["original_embedding_name"].apply(lambda x: x if x.endswith(".npy") else f"{x}.npy")
                filtered_set = set(zip(meta["class_name"].values, meta["original_embedding_name"].values))
                for cat in categories_to_plot:
                    cat_sim = sim_df[sim_df["class_name"].astype(str).str.lower() == str(cat).lower()].copy()
                    if "embedding_name" in cat_sim.columns:
                        cat_sim = cat_sim[cat_sim.apply(lambda r: (r["class_name"], r["embedding_name"]) in filtered_set, axis=1)]
                    ranked = cat_sim.sort_values("sim_value", ascending=False)
                    tsne_ranked_stems[cat] = [str(r["embedding_name"]).rstrip(".npy") for _, r in ranked.iterrows()]

        for cat in categories_to_plot:
            match = next((x for x in categories_common if x.lower() == cat.lower()), None)
            if not match:
                continue
            bv_X = bv_embeddings[match]
            bv_id_list = bv_ids[match]
            # Strict mode: use only per-file precision-valid exemplars for this category
            # for every embedding model (CLIP, DINOv3, ...).
            valid_stems_for_cat = set(valid_file_stems.get(match.lower(), set()))
            keep_idx = [i for i, _id in enumerate(bv_id_list) if str(_id[2]).strip() in valid_stems_for_cat]
            if len(keep_idx) < 2:
                print(f"[{embedding}] Skipping {match}: <2 valid exemplars after per-file precision filter.")
                continue
            bv_X = bv_X[keep_idx]
            rank_group = category_to_rank_group.get(match, category_to_rank_group.get(match.lower(), "selected"))
            out_path = tsne_dir / f"tsne_{rank_group}_{match}.png"
            title_suffix = f" ({rank_group})"

            bv_centroid = bv_X.mean(axis=0)
            dist_bv_to_bv = np.linalg.norm(bv_X - bv_centroid, axis=1)

            plot_tsne_one_category(
                category=match,
                bv_X=bv_X,
                bv_centroid=bv_centroid,
                dist_bv_to_bv=dist_bv_to_bv,
                out_path=out_path,
                perplexity=tsne_perplexity,
                zscore=tsne_zscore,
                title_suffix=title_suffix,
                model_label=embedding,
            )
            print(f"[{embedding}] Saved t-SNE: {out_path}")

        def _tsne_bundle_for_category(match):
            bv_X = bv_embeddings[match]
            bv_id_list = bv_ids[match]
            valid_stems_for_cat = set(valid_file_stems.get(match.lower(), set()))
            keep_idx = [i for i, _id in enumerate(bv_id_list) if str(_id[2]).strip() in valid_stems_for_cat]
            if len(keep_idx) < 2:
                return None
            bv_X = bv_X[keep_idx]
            bv_centroid = bv_X.mean(axis=0)
            dist_bv_to_bv = np.linalg.norm(bv_X - bv_centroid, axis=1)
            return {"name": match, "bv_X": bv_X, "bv_centroid": bv_centroid, "dist_bv_to_bv": dist_bv_to_bv}

        joint_bg_bundles = None
        _joint_only_local = globals().get("tsne_joint_only_local", False)
        if tsne_joint_multi_category and tsne_joint_tsne_background == "all_valid_categories" and not _joint_only_local:
            joint_bg_bundles = []
            for match in categories_common:
                b = _tsne_bundle_for_category(match)
                if b is not None:
                    joint_bg_bundles.append({"name": b["name"], "bv_X": b["bv_X"]})
            if joint_bg_bundles:
                print(
                    f"[{embedding}] Joint t-SNE background: {len(joint_bg_bundles)} categories, "
                    f"{sum(b['bv_X'].shape[0] for b in joint_bg_bundles)} exemplars (fit on all; plot highlights only)."
                )
            else:
                joint_bg_bundles = None

        if tsne_joint_multi_category and len(categories_to_plot) >= 2:
            _joint_save_emb = globals().get("tsne_joint_save_embeddings")
            if _joint_save_emb is None or len(_joint_save_emb) == 0:
                _joint_save_emb = list(embeddings_to_run)
            _joint_color_mode = globals().get("tsne_joint_color_mode", "magma")
            if tsne_joint_category_groups is None:
                if tsne_all_categories:
                    print(
                        f"[{embedding}] Joint t-SNE: skipping auto panel when tsne_all_categories=True "
                        "(set tsne_joint_category_groups to a list of groups, e.g. [['cup','book','watch']])."
                    )
                    joint_groups = []
                else:
                    joint_groups = [categories_to_plot[: min(4, len(categories_to_plot))]]
            else:
                joint_groups = list(tsne_joint_category_groups)
            if joint_groups and not _joint_tsne_groups_cache:
                _joint_tsne_groups_cache[:] = joint_groups
            for gi, group in enumerate(joint_groups):
                if embedding not in _joint_save_emb:
                    continue
                specs = []
                for cat in group:
                    match = next((x for x in categories_common if x.lower() == str(cat).strip().lower()), None)
                    if not match:
                        continue
                    b = _tsne_bundle_for_category(match)
                    if b is not None:
                        specs.append(b)
                if len(specs) < 2:
                    print(f"[{embedding}] Skipping joint t-SNE panel {gi}: fewer than 2 categories with >=2 valid exemplars")
                    continue
                if len(specs) > 4:
                    print(f"[{embedding}] Joint panel {gi}: truncating to first 4 categories (got {len(specs)})")
                    specs = specs[:4]
                slug = "+".join(s["name"].replace(" ", "_") for s in specs)
                jpath = tsne_dir / f"tsne_joint_{slug}.png"
                jpath_local = tsne_dir / f"tsne_joint_{slug}_local.png"
                if not _joint_only_local:
                    plot_tsne_multi_categories(
                        category_specs=specs,
                        out_path=jpath,
                        perplexity=tsne_perplexity,
                        zscore=tsne_zscore,
                        title_suffix="",
                        model_label=embedding,
                        background_bundles=joint_bg_bundles,
                        joint_color_mode=_joint_color_mode,
                    )
                    print(f"[{embedding}] Saved joint t-SNE: {jpath}")
                if joint_bg_bundles is not None:
                    plot_tsne_multi_categories(
                        category_specs=specs,
                        out_path=jpath_local,
                        perplexity=tsne_perplexity,
                        zscore=tsne_zscore,
                        title_suffix=" | local t-SNE (highlighted cats only)",
                        model_label=embedding,
                        background_bundles=None,
                        joint_color_mode=_joint_color_mode,
                    )
                    print(f"[{embedding}] Saved joint t-SNE (local fit): {jpath_local}")
                elif _joint_only_local:
                    plot_tsne_multi_categories(
                        category_specs=specs,
                        out_path=jpath_local,
                        perplexity=tsne_perplexity,
                        zscore=tsne_zscore,
                        title_suffix=" | local t-SNE (highlighted cats only)",
                        model_label=embedding,
                        background_bundles=None,
                        joint_color_mode=_joint_color_mode,
                    )
                    print(f"[{embedding}] Saved joint t-SNE (local only): {jpath_local}")
else:
    print("t-SNE generation is disabled by config.")

[dinov3] Using Plot A selected categories: ['watch', 'shower', 'bird', 'cup', 'book']
[dinov3] Running t-SNE for 5 categories: ['watch', 'shower', 'bird', 'cup', 'book']
[dinov3] t-SNE output directory: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_tsne_by_category
  Saved t-SNE: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_tsne_by_category/tsne_plotA_watch.png
  Saved t-SNE PDF: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_tsne_by_category/tsne_plotA_watch.pdf
[dinov3] Saved t-SNE: /home/j7yang/babyview-projects/vss2026/object-detection/analysis/ccn-2026/plotB_tsne_distance_to_centroid_outputs_20260402/bv-to-bv-centroid_distance_dinov3_tsne_by_category/tsn

## Optional: Montages (disabled here)

Placeholder for top-12 BV by similarity (all-frames CSV) or farthest from **BV** centroid; the next cell only prints that montages are off for this notebook.

In [13]:
print("Montage generation is disabled in notebook 2 (t-SNE only).")

Montage generation is disabled in notebook 2 (t-SNE only).


## CLIP vs DINOv3 comparison (mean BV→BV centroid per category)

In [14]:
if set(["clip", "dinov3"]).issubset(results_by_embedding.keys()):
    clip_df = results_by_embedding["clip"]["summary_df"].copy()
    dino_df = results_by_embedding["dinov3"]["summary_df"].copy()

    id_col = "n_bv_exemplars"
    if id_col not in clip_df.columns or id_col not in dino_df.columns:
        if "n_bv" in clip_df.columns and "n_bv" in dino_df.columns:
            id_col = "n_bv"
        else:
            raise ValueError(
                "Expected exemplar count column in both summary dataframes. "
                f"clip columns={list(clip_df.columns)}, dinov3 columns={list(dino_df.columns)}"
            )

    keep_cols = ["category", "mean_bv_to_bv_centroid", id_col]
    clip_df = clip_df[keep_cols].rename(columns={
        "mean_bv_to_bv_centroid": "clip_mean_bv_to_bv_centroid",
        id_col: "clip_n_bv_exemplars",
    })
    dino_df = dino_df[keep_cols].rename(columns={
        "mean_bv_to_bv_centroid": "dinov3_mean_bv_to_bv_centroid",
        id_col: "dinov3_n_bv_exemplars",
    })

    comparison_df = clip_df.merge(dino_df, on="category", how="inner")
    comparison_df["delta_bv_centroid_dinov3_minus_clip"] = (
        comparison_df["dinov3_mean_bv_to_bv_centroid"] - comparison_df["clip_mean_bv_to_bv_centroid"]
    )

    corr_bv = comparison_df[["clip_mean_bv_to_bv_centroid", "dinov3_mean_bv_to_bv_centroid"]].corr().iloc[0, 1]

    cmp_path = out_dir / f"{output_metric_slug}_clip_vs_dinov3_comparison.csv"
    comparison_df.sort_values("delta_bv_centroid_dinov3_minus_clip", ascending=False).to_csv(cmp_path, index=False)
    print(f"Saved model comparison table: {cmp_path}")
    print(f"Category count: {len(comparison_df)}")
    print(f"Correlation (BV->BV centroid): {corr_bv:.3f}")

    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

    axes[0].scatter(
        comparison_df["clip_mean_bv_to_bv_centroid"],
        comparison_df["dinov3_mean_bv_to_bv_centroid"],
        s=18, alpha=0.8
    )
    mn = min(comparison_df["clip_mean_bv_to_bv_centroid"].min(), comparison_df["dinov3_mean_bv_to_bv_centroid"].min())
    mx = max(comparison_df["clip_mean_bv_to_bv_centroid"].max(), comparison_df["dinov3_mean_bv_to_bv_centroid"].max())
    axes[0].plot([mn, mx], [mn, mx], linestyle="--", linewidth=1)
    axes[0].set_xlabel("CLIP mean BV->BV centroid")
    axes[0].set_ylabel("DINOv3 mean BV->BV centroid")
    axes[0].set_title(f"Centroid distance agreement (r={corr_bv:.2f})")

    axes[1].hist(comparison_df["delta_bv_centroid_dinov3_minus_clip"], bins=25, alpha=0.9)
    axes[1].axvline(0.0, linestyle="--", linewidth=1)
    axes[1].set_xlabel("DINOv3 - CLIP (mean BV->BV centroid)")
    axes[1].set_ylabel("Category count")
    axes[1].set_title("Cross-model difference distribution")

    fig.tight_layout()
    fig_path = out_dir / f"{output_metric_slug}_clip_vs_dinov3_comparison.png"
    fig_pdf_path = fig_path.with_suffix(".pdf")
    fig.savefig(fig_path, bbox_inches="tight")
    fig.savefig(fig_pdf_path, bbox_inches="tight")
    plt.show()
    print(f"Saved comparison figure: {fig_path}")
    print(f"Saved comparison PDF: {fig_pdf_path}")
else:
    print("Skipping model comparison: need both clip and dinov3 in results_by_embedding")

Skipping model comparison: need both clip and dinov3 in results_by_embedding


## Side-by-side t-SNE (CLIP vs DINOv3): single-category panels and joint multi-category panels

In [15]:
combined_tsne_categories = ["bird", "book", "watch", "pony"]
_joint_tsne_groups_cache = globals().get("_joint_tsne_groups_cache", [])
tsne_joint_multi_category = globals().get("tsne_joint_multi_category", False)

if set(["clip", "dinov3"]).issubset(results_by_embedding.keys()):
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt
    from matplotlib.collections import PathCollection

    def _tsne_coords_for_panel(bv_X, bv_centroid, perplexity=30, random_state=42, zscore=True):
        n_bv = bv_X.shape[0]
        X_all = np.vstack([bv_X, bv_centroid.reshape(1, -1)])
        if zscore:
            mean, std = X_all.mean(axis=0), X_all.std(axis=0)
            std = np.where(std > 1e-8, std, 1.0)
            X_all = (X_all - mean) / std
        n_all = X_all.shape[0]
        p = min(perplexity, max(2, (n_all - 1) // 3))
        tsne = TSNE(
            n_components=2,
            perplexity=p,
            random_state=random_state,
            init="pca",
            learning_rate="auto",
            max_iter=1500,
        )
        coords = tsne.fit_transform(X_all)
        bx, by = coords[:n_bv].T
        cx, cy = coords[n_bv]
        return bx, by, cx, cy

    combined_dir = out_dir / f"{output_metric_slug}_clip_vs_dinov3_tsne_side_by_side"
    combined_dir.mkdir(exist_ok=True)
    print(f"Combined t-SNE output directory: {combined_dir}")

    for cat in combined_tsne_categories:
        cat_norm = str(cat).strip().lower()
        panel_data = {}

        for embedding, model_name in [("clip", "CLIP"), ("dinov3", "DINOv3")]:
            categories_common = results_by_embedding[embedding]["categories_common"]
            match = next((x for x in categories_common if x.lower() == cat_norm), None)
            if not match:
                print(f"[{embedding}] Skipping {cat}: category not found.")
                continue

            bv_X = results_by_embedding[embedding]["bv_embeddings"][match]
            bv_id_list = results_by_embedding[embedding]["bv_ids"][match]

            valid_stems_for_cat = set(valid_file_stems.get(match.lower(), set()))
            keep_idx = [i for i, _id in enumerate(bv_id_list) if str(_id[2]).strip() in valid_stems_for_cat]
            if len(keep_idx) < 2:
                print(f"[{embedding}] Skipping {cat}: <2 valid exemplars after per-file precision filter.")
                continue

            bv_X = bv_X[keep_idx]
            bv_centroid = bv_X.mean(axis=0)
            dist_bv_to_bv = np.linalg.norm(bv_X - bv_centroid, axis=1)
            bx, by, cx, cy = _tsne_coords_for_panel(
                bv_X,
                bv_centroid,
                perplexity=tsne_perplexity,
                random_state=42,
                zscore=tsne_zscore,
            )
            panel_data[embedding] = {
                "model_name": model_name,
                "bx": bx,
                "by": by,
                "cx": cx,
                "cy": cy,
                "dist": dist_bv_to_bv,
                "n": len(keep_idx),
                "label_category": match,
            }

        if set(["clip", "dinov3"]).difference(panel_data.keys()):
            print(f"Skipping {cat}: need both CLIP and DINOv3 panels after filtering.")
            continue

        d_clip = panel_data["clip"]["dist"]
        d_dino = panel_data["dinov3"]["dist"]
        d_min = float(min(np.min(d_clip), np.min(d_dino)))
        d_max = float(max(np.max(d_clip), np.max(d_dino)))
        if d_max - d_min < 1e-12:
            d_max = d_min + 1.0

        plt.style.use("seaborn-v0_8-white")
        fig, axes = plt.subplots(1, 2, figsize=(14, 6.2), facecolor="#fbfbfc")
        for ax, embedding in zip(axes, ["clip", "dinov3"]):
            p = panel_data[embedding]
            sc = ax.scatter(
                p["bx"],
                p["by"],
                c=p["dist"],
                s=58,
                cmap="magma",
                vmin=d_min,
                vmax=d_max,
                alpha=0.95,
                edgecolors="white",
                linewidths=0.45,
                label=f"{p['model_name']} exemplars (n={p['n']})",
                zorder=3,
                rasterized=False,
            )
            _tsne_colorbar(ax, sc, "Distance to BV centroid")

            ax.scatter([p["cx"]], [p["cy"]], s=_JOINT_CENTROID_S_FILLED, marker="o", c="#1958d1", edgecolors="white", linewidths=1.5, label="BV centroid", zorder=6, rasterized=False)
            ax.scatter([p["cx"]], [p["cy"]], s=_JOINT_CENTROID_S_RING, marker="o", facecolors="none", edgecolors="#1958d1", linewidths=1.5, alpha=0.45, zorder=5, rasterized=False)

            _tsne_style_axes(
                ax,
                f"{p['label_category']} | t-SNE ({p['model_name']})",
                "t-SNE 1",
                "t-SNE 2",
            )
            legend = ax.legend(loc="best", frameon=True, markerscale=1.0, prop={"weight": "bold", "size": _TSNE_LEGEND_FS})
            for h in legend.legend_handles:
                if isinstance(h, PathCollection):
                    h.set_sizes([58])
            ax.set_aspect("equal")
            ax.grid(False)
            for spine in ["top", "right"]:
                ax.spines[spine].set_visible(False)

        fig.suptitle(f"{cat_norm} | CLIP vs DINOv3 t-SNE (precision-valid exemplars)", fontsize=16, fontweight="bold", y=1.02)
        fig.tight_layout()
        out_png = combined_dir / f"tsne_side_by_side_{cat_norm}.png"
        out_pdf = out_png.with_suffix(".pdf")
        fig.savefig(out_png, dpi=190, facecolor=fig.get_facecolor(), bbox_inches="tight")
        fig.savefig(out_pdf, format="pdf", bbox_inches="tight")
        plt.close(fig)
        print(f"Saved combined t-SNE: {out_png}")
        print(f"Saved combined t-SNE PDF: {out_pdf}")

        # Additional version with aligned axes across CLIP and DINOv3 panels.
        all_x = np.concatenate([
            panel_data["clip"]["bx"], [panel_data["clip"]["cx"]],
            panel_data["dinov3"]["bx"], [panel_data["dinov3"]["cx"]],
        ])
        all_y = np.concatenate([
            panel_data["clip"]["by"], [panel_data["clip"]["cy"]],
            panel_data["dinov3"]["by"], [panel_data["dinov3"]["cy"]],
        ])
        x_min, x_max = float(np.min(all_x)), float(np.max(all_x))
        y_min, y_max = float(np.min(all_y)), float(np.max(all_y))
        x_pad = max((x_max - x_min) * 0.08, 1e-6)
        y_pad = max((y_max - y_min) * 0.08, 1e-6)

        fig2, axes2 = plt.subplots(1, 2, figsize=(14, 6.2), facecolor="#fbfbfc")
        for ax, embedding in zip(axes2, ["clip", "dinov3"]):
            p = panel_data[embedding]
            sc = ax.scatter(
                p["bx"],
                p["by"],
                c=p["dist"],
                s=58,
                cmap="magma",
                vmin=d_min,
                vmax=d_max,
                alpha=0.95,
                edgecolors="white",
                linewidths=0.45,
                label=f"{p['model_name']} exemplars (n={p['n']})",
                zorder=3,
                rasterized=False,
            )
            _tsne_colorbar(ax, sc, "Distance to BV centroid")
            ax.scatter([p["cx"]], [p["cy"]], s=_JOINT_CENTROID_S_FILLED, marker="o", c="#1958d1", edgecolors="white", linewidths=1.5, label="BV centroid", zorder=6, rasterized=False)
            ax.scatter([p["cx"]], [p["cy"]], s=_JOINT_CENTROID_S_RING, marker="o", facecolors="none", edgecolors="#1958d1", linewidths=1.5, alpha=0.45, zorder=5, rasterized=False)
            ax.set_xlim(x_min - x_pad, x_max + x_pad)
            ax.set_ylim(y_min - y_pad, y_max + y_pad)
            _tsne_style_axes(
                ax,
                f"{p['label_category']} | t-SNE ({p['model_name']})",
                "t-SNE 1",
                "t-SNE 2",
            )
            legend = ax.legend(loc="best", frameon=True, markerscale=1.0, prop={"weight": "bold", "size": _TSNE_LEGEND_FS})
            for h in legend.legend_handles:
                if isinstance(h, PathCollection):
                    h.set_sizes([58])
            ax.set_aspect("equal")
            ax.grid(False)
            for spine in ["top", "right"]:
                ax.spines[spine].set_visible(False)

        fig2.suptitle(f"{cat_norm} | CLIP vs DINOv3 t-SNE (aligned axes)", fontsize=16, fontweight="bold", y=1.02)
        fig2.tight_layout()
        out_aligned_png = combined_dir / f"tsne_side_by_side_aligned_{cat_norm}.png"
        out_aligned_pdf = out_aligned_png.with_suffix(".pdf")
        fig2.savefig(out_aligned_png, dpi=190, facecolor=fig2.get_facecolor(), bbox_inches="tight")
        fig2.savefig(out_aligned_pdf, format="pdf", bbox_inches="tight")
        plt.close(fig2)
        print(f"Saved combined aligned t-SNE: {out_aligned_png}")
        print(f"Saved combined aligned t-SNE PDF: {out_aligned_pdf}")

    _joint_groups_for_sbs = globals().get("_joint_tsne_groups_cache") or []
    if (
        globals().get("tsne_joint_multi_category", False)
        and _joint_groups_for_sbs
        and globals().get("tsne_joint_clip_vs_dinov3_side_by_side", True)
    ):
        joint_combined_dir = out_dir / f"{output_metric_slug}_clip_vs_dinov3_tsne_joint_side_by_side"
        joint_combined_dir.mkdir(exist_ok=True)
        print(f"Joint multi-category side-by-side directory: {joint_combined_dir}")

        def _joint_bundle_from_results(embedding_key, match):
            bv_X = results_by_embedding[embedding_key]["bv_embeddings"][match]
            bv_id_list = results_by_embedding[embedding_key]["bv_ids"][match]
            valid_stems_for_cat = set(valid_file_stems.get(match.lower(), set()))
            keep_idx = [i for i, _id in enumerate(bv_id_list) if str(_id[2]).strip() in valid_stems_for_cat]
            if len(keep_idx) < 2:
                return None
            bv_X = bv_X[keep_idx]
            bv_centroid = bv_X.mean(axis=0)
            dist_bv_to_bv = np.linalg.norm(bv_X - bv_centroid, axis=1)
            return {"name": match, "bv_X": bv_X, "bv_centroid": bv_centroid, "dist_bv_to_bv": dist_bv_to_bv}

        _jbg_mode = globals().get("tsne_joint_tsne_background", "all_valid_categories")
        _joint_bg_clip = _joint_bg_dino = None
        if _jbg_mode == "all_valid_categories":
            _joint_bg_clip, _joint_bg_dino = [], []
            for m in results_by_embedding["clip"]["categories_common"]:
                bc = _joint_bundle_from_results("clip", m)
                if bc is not None:
                    _joint_bg_clip.append({"name": bc["name"], "bv_X": bc["bv_X"]})
            for m in results_by_embedding["dinov3"]["categories_common"]:
                bd = _joint_bundle_from_results("dinov3", m)
                if bd is not None:
                    _joint_bg_dino.append({"name": bd["name"], "bv_X": bd["bv_X"]})
            if not _joint_bg_clip or not _joint_bg_dino:
                _joint_bg_clip = _joint_bg_dino = None
            elif _joint_bg_clip and _joint_bg_dino:
                print(
                    f"Joint side-by-side background: clip {len(_joint_bg_clip)} cats / "
                    f"{sum(b['bv_X'].shape[0] for b in _joint_bg_clip)} ex., "
                    f"dinov3 {len(_joint_bg_dino)} cats / {sum(b['bv_X'].shape[0] for b in _joint_bg_dino)} ex."
                )

        for gi, group in enumerate(_joint_groups_for_sbs):
            specs_clip, specs_dino = [], []
            for cat in group:
                cat_norm = str(cat).strip().lower()
                m_clip = next(
                    (x for x in results_by_embedding["clip"]["categories_common"] if x.lower() == cat_norm), None
                )
                m_dino = next(
                    (x for x in results_by_embedding["dinov3"]["categories_common"] if x.lower() == cat_norm), None
                )
                if not m_clip or not m_dino:
                    continue
                bc = _joint_bundle_from_results("clip", m_clip)
                bd = _joint_bundle_from_results("dinov3", m_dino)
                if bc is not None and bd is not None:
                    specs_clip.append(bc)
                    specs_dino.append(bd)
            if len(specs_clip) < 2:
                print(
                    f"Skipping joint side-by-side panel {gi}: fewer than 2 categories with "
                    f">=2 valid exemplars in both models."
                )
                continue
            if len(specs_clip) > 4:
                specs_clip = specs_clip[:4]
                specs_dino = specs_dino[:4]
            slug = "+".join(s["name"].replace(" ", "_") for s in specs_clip)
            out_j = joint_combined_dir / f"tsne_joint_side_by_side_{slug}.png"
            plot_tsne_multi_categories_side_by_side(
                specs_clip,
                specs_dino,
                out_j,
                perplexity=tsne_perplexity,
                random_state=42,
                zscore=tsne_zscore,
                title_suffix="",
                background_bundles_clip=_joint_bg_clip,
                background_bundles_dinov3=_joint_bg_dino,
                joint_color_mode=globals().get("tsne_joint_color_mode", "magma"),
            )
            if _joint_bg_clip is not None and _joint_bg_dino is not None:
                out_j_local = joint_combined_dir / f"tsne_joint_side_by_side_{slug}_local.png"
                plot_tsne_multi_categories_side_by_side(
                    specs_clip,
                    specs_dino,
                    out_j_local,
                    perplexity=tsne_perplexity,
                    random_state=42,
                    zscore=tsne_zscore,
                    title_suffix=" | local t-SNE (highlighted cats only)",
                    background_bundles_clip=None,
                    background_bundles_dinov3=None,
                    joint_color_mode=globals().get("tsne_joint_color_mode", "magma"),
                )
    elif globals().get("tsne_joint_multi_category", False):
        print(
            "Joint multi-category CLIP vs DINO side-by-side skipped: "
            "tsne_joint_clip_vs_dinov3_side_by_side is False, or run the t-SNE cell first so "
            "_joint_tsne_groups_cache is populated, or set tsne_joint_category_groups."
        )
else:
    print("Skipping side-by-side t-SNE: need both clip and dinov3 in results_by_embedding")


Skipping side-by-side t-SNE: need both clip and dinov3 in results_by_embedding
